In [3]:
# ==============================
# ReviewGuard: Preprocessing
# ==============================

import pandas as pd
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# ==============================
# 1️⃣ Load Dataset
# ==============================

df = pd.read_csv("../data/raw/cleaned_base_deceptive_reviews.csv")

# Keep only required columns
df = df[['Review', 'Label']]
df.columns = ['review', 'label']

print("Dataset loaded:", df.shape)
df.head()

# ==============================
# 2️⃣ Encode Labels
# ==============================

df['label'] = df['label'].map({
    'truthful': 0,
    'deceptive': 1
})

df['label'].value_counts()

# ==============================
# 3️⃣ Clean Text
# ==============================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text.strip()

df['cleaned_review'] = df['review'].apply(clean_text)
df.head()

# ==============================
# 4️⃣ Train-Test Split
# ==============================

X = df['cleaned_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# ==============================
# 5️⃣ TF-IDF Vectorization
# ==============================

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)

# ==============================
# 6️⃣ Save Preprocessed Objects
# ==============================

os.makedirs("../models", exist_ok=True)

joblib.dump(vectorizer, "../models/vectorizer.pkl")
joblib.dump(X_train_tfidf, "../models/X_train.pkl")
joblib.dump(X_test_tfidf, "../models/X_test.pkl")
joblib.dump(y_train, "../models/y_train.pkl")
joblib.dump(y_test, "../models/y_test.pkl")

print("✅ Preprocessing complete. Files saved in models/")


Dataset loaded: (1600, 2)
Train size: (1280,)
Test size: (320,)
TF-IDF shape: (1280, 5000)
✅ Preprocessing complete. Files saved in models/
